In [1]:
# ============ 셀 1: 설정 (Muon 학습률 재스윕) ============
import os, json, time, math, random, glob, shutil
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

assert torch.cuda.is_available(), "GPU가 꺼져 있어요. Settings → Accelerator → GPU T4 로 바꾸세요."
DEV = 'cuda'

OUT  = '/kaggle/working/exp_llm_muon_lrsweep2'
DATA = '/tmp/llm_data'
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)

CFG = dict(
    seed=0, ctx=256, vocab=50304,
    d_model=512, n_layer=6, n_head=8,
    # 사전학습: Muon(블록 안 행렬) + AdamW(임베딩·LayerNorm·bias), WSD — L2와 동일
    pre_epochs=10, pre_bs=32, pre_wd=0.1, warmup_frac=0.02, decay_frac=0.2,
    sweep_lrs=[0.0025, 0.005, 0.01], sweep_frac=0.25, grad_clip=1.0,   # ← 어제보다 낮은 쪽으로
    adam_lr_aux=1e-3,
    muon_momentum=0.95, ns_steps=5,
    eval_seqs=400, eval_bs=32,
)
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print('GPU :', torch.cuda.get_device_name(0), '| torch', torch.__version__)

GPU : Tesla T4 | torch 2.10.0+cu128


In [2]:
# ============ 셀 2: BabyLM 다운로드, 토큰화 (L1·L2와 같은 방식) ============
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2'); tok.model_max_length = int(1e9)
EOS = tok.eos_token_id

def encode_texts(texts, bs=256):
    parts = []
    for i in range(0, len(texts), bs):
        for ids in tok(texts[i:i + bs])['input_ids']:
            parts.append(np.array(ids + [EOS], dtype=np.uint16))
    return np.concatenate(parts)

def chunk_lines(text, n=200):
    lines = text.split('\n')
    return ['\n'.join(lines[i:i + n]) for i in range(0, len(lines), n)]

if os.path.exists(f'{DATA}/babylm_val.npy'):
    print('BabyLM 토큰 파일이 이미 있음 → 건너뜀')
else:
    try:
        from huggingface_hub import snapshot_download
        root = snapshot_download('cambridge-climb/BabyLM', repo_type='dataset', allow_patterns=['10M/*', 'dev/*'])
        tr_files = sorted(f for f in glob.glob(f'{root}/10M/*') if os.path.isfile(f))
        dv_files = sorted(f for f in glob.glob(f'{root}/dev/*') if os.path.isfile(f))
        assert tr_files and dv_files, '파일 없음'
        tr_texts, dv_texts = [], []
        for f in tr_files: tr_texts += chunk_lines(open(f, encoding='utf-8', errors='ignore').read())
        for f in dv_files: dv_texts += chunk_lines(open(f, encoding='utf-8', errors='ignore').read()[:500_000])
        src = 'cambridge-climb/BabyLM (2023)'
    except Exception as e:
        print('2023판 다운로드 실패 → 2026판으로 대체:', repr(e)[:200])
        from datasets import load_dataset
        ds = load_dataset('BabyLM-community/BabyLM-2026-Strict-Small', split='train')
        col = [c for c, t in ds.features.items() if getattr(t, 'dtype', None) == 'string'][0]
        lines = [t for t in ds[col] if t and t.strip()]
        idx = np.random.RandomState(0).permutation(len(lines)); nv = len(lines) // 20
        dv_lines = [lines[i] for i in idx[:nv]]; tr_lines = [lines[i] for i in idx[nv:]]
        tr_texts = ['\n'.join(tr_lines[i:i + 200]) for i in range(0, len(tr_lines), 200)]
        dv_texts = ['\n'.join(dv_lines[i:i + 200]) for i in range(0, len(dv_lines), 200)]
        src = 'BabyLM-community/BabyLM-2026-Strict-Small (5%를 검증용으로 분리)'
    print('BabyLM 출처:', src, flush=True)
    tr = encode_texts(tr_texts); dv = encode_texts(dv_texts)
    np.save(f'{DATA}/babylm_train.npy', tr); np.save(f'{DATA}/babylm_val.npy', dv)
    print(f'BabyLM 토큰: train {len(tr)/1e6:.1f}M, val {len(dv)/1e6:.1f}M   (L1·L2: 15.7M / 0.8M)')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

open_subtitles.train.txt:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

simple_wiki.train.txt: 0.00B [00:00, ?B/s]

switchboard.train.txt: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1104106 [00:00<?, ? examples/s]

BabyLM 출처: BabyLM-community/BabyLM-2026-Strict-Small (5%를 검증용으로 분리)
BabyLM 토큰: train 15.7M, val 0.8M   (L1·L2: 15.7M / 0.8M)


In [3]:
# ============ 셀 3: 데이터 스트림, 모델, Muon, 학습 루프 (L2와 동일) ============
def load_tokens(name):
    return torch.from_numpy(np.load(f'{DATA}/{name}.npy').astype(np.int64)).to(DEV)

TR_BABY, VA_BABY = load_tokens('babylm_train'), load_tokens('babylm_val')

class Stream:
    def __init__(self, tokens, bs, ctx=CFG['ctx']):
        n = (len(tokens) - 1) // ctx
        self.chunks = tokens[:n * ctx + 1].unfold(0, ctx + 1, ctx)
        self.n, self.bs, self.spe = n, bs, n // bs
        self._key = None
    def batch(self, step, seed):
        ep, i = divmod(step, self.spe)
        if self._key != (seed, ep):
            g = torch.Generator().manual_seed(seed * 100003 + ep)
            self._perm = torch.randperm(self.n, generator=g).to(DEV); self._key = (seed, ep)
        b = self.chunks[self._perm[i * self.bs:(i + 1) * self.bs]]
        return b[:, :-1], b[:, 1:]

def make_eval(tokens, n, seed, ctx=CFG['ctx']):
    starts = np.random.RandomState(seed).randint(0, len(tokens) - ctx - 1, size=n)
    idx = torch.from_numpy(starts).to(DEV)[:, None] + torch.arange(ctx + 1, device=DEV)
    b = tokens[idx]
    return b[:, :-1], b[:, 1:]

EV_BABY = make_eval(VA_BABY, CFG['eval_seqs'], seed=11)    # L1·L2와 같은 평가 문장

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.ln1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d); self.fc = nn.Linear(d, 4 * d); self.fc2 = nn.Linear(4 * d, d)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(y.transpose(1, 2).contiguous().view(B, T, C))
        return x + self.fc2(F.gelu(self.fc(self.ln2(x))))

class GPT(nn.Module):
    def __init__(self, V=CFG['vocab'], d=CFG['d_model'], L=CFG['n_layer'], h=CFG['n_head'], T=CFG['ctx']):
        super().__init__()
        self.wte = nn.Embedding(V, d); self.wpe = nn.Embedding(T, d)
        self.blocks = nn.ModuleList([Block(d, h) for _ in range(L)])
        self.lnf = nn.LayerNorm(d)
        for n, p in self.named_parameters():
            if p.ndim >= 2:
                std = 0.02 / math.sqrt(2 * L) if n.endswith(('proj.weight', 'fc2.weight')) else 0.02
                nn.init.normal_(p, 0.0, std)
            elif n.endswith('bias'):
                nn.init.zeros_(p)
    def forward(self, idx, targets):
        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks:
            x = b(x)
        logits = F.linear(self.lnf(x), self.wte.weight)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))

def make_model():
    return GPT().to(DEV)

# ---------- Muon: 업데이트 방향을 직교화(Newton-Schulz)해서 쓰는 옵티마이저 ----------
def newton_schulz5(G, steps):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + 1e-7)
    tr = X.size(0) > X.size(1)
    if tr: X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    return X.T if tr else X

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps))
    @torch.no_grad()
    def step(self, closure=None):
        for grp in self.param_groups:
            for p in grp['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'buf' not in st: st['buf'] = torch.zeros_like(p.grad)
                buf = st['buf']; buf.mul_(grp['momentum']).add_(p.grad)
                u = p.grad.add(buf, alpha=grp['momentum']) if grp['nesterov'] else buf
                u = newton_schulz5(u, grp['ns_steps'])
                u *= max(1.0, p.size(0) / p.size(1)) ** 0.5
                p.add_(u.to(p.dtype), alpha=-grp['lr'])

def make_muon_opts(model, muon_lr, adam_lr, wd):
    """블록 안 2차원 행렬 → Muon, 나머지(임베딩·LayerNorm·bias) → AdamW"""
    hidden = [p for p in model.blocks.parameters() if p.ndim == 2]
    hid = {id(p) for p in hidden}
    rest = [p for p in model.parameters() if id(p) not in hid]
    muon = Muon(hidden, lr=muon_lr, momentum=CFG['muon_momentum'], ns_steps=CFG['ns_steps'])
    adam = torch.optim.AdamW([{'params': [p for p in rest if p.ndim >= 2], 'weight_decay': wd},
                              {'params': [p for p in rest if p.ndim < 2], 'weight_decay': 0.0}],
                             lr=adam_lr, betas=(0.9, 0.95), eps=1e-8)
    for o, lr in ((muon, muon_lr), (adam, adam_lr)):
        for g in o.param_groups: g['base_lr'] = lr
    return [muon, adam]

def wsd_fn(total, warm, decay_start):
    def f(s):
        if s < warm: return (s + 1) / warm
        if s < decay_start: return 1.0
        return max(0.0, (total - s) / (total - decay_start))
    return f

def train_step(model, opts, scaler, x, y):
    with torch.autocast('cuda', dtype=torch.float16):
        loss = model(x, y)
    model.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    for o in opts: scaler.unscale_(o)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
    for o in opts: scaler.step(o)
    scaler.update()
    return loss.detach()

@torch.no_grad()
def eval_loss(model, ev):
    model.eval(); X, Y = ev; tot = 0.0; bs = CFG['eval_bs']
    for i in range(0, len(X), bs):
        with torch.autocast('cuda', dtype=torch.float16):
            tot += model(X[i:i + bs], Y[i:i + bs]).float().item() * len(X[i:i + bs])
    model.train()
    return tot / len(X)

def train_steps(model, opts, scaler, stream, s_from, s_to, lr_fn, seed, log, eval_every, tag, total):
    model.train(); t0 = time.time(); acc = torch.zeros((), device=DEV); cnt = 0
    for s in range(s_from, s_to):
        f = lr_fn(s)
        for o in opts:
            for g in o.param_groups: g['lr'] = g['base_lr'] * f
        x, y = stream.batch(s, seed)
        acc += train_step(model, opts, scaler, x, y); cnt += 1
        if (s + 1) % eval_every == 0 or s + 1 == s_to:
            vl = eval_loss(model, EV_BABY); tl = (acc / cnt).item(); dt = time.time() - t0
            log.append(dict(step=s + 1, train_loss=tl, val_loss=vl, sec=dt))
            print(f'[{tag}] step {s+1}/{total}  train {tl:.3f}  BabyLM val {vl:.3f} (ppl {math.exp(min(vl, 20)):.1f})  '
                  f'{dt/cnt:.3f}s/step', flush=True)
            acc.zero_(); cnt = 0; t0 = time.time()

tr_stream = Stream(TR_BABY, CFG['pre_bs'])
TOTAL = CFG['pre_epochs'] * tr_stream.spe
print(f'epoch당 {tr_stream.spe} step, 사전학습 전체 {TOTAL} step (L2: 1914 / 19140)')

epoch당 1914 step, 사전학습 전체 19140 step (L2: 1914 / 19140)


In [4]:
# ============ 셀 4: Muon 학습률 재스윕 (전체의 25% 길이, 보조 AdamW lr은 1e-3 고정) ============
sweep_path = f'{OUT}/lr_sweep.json'
SWEEP = json.load(open(sweep_path)) if os.path.exists(sweep_path) else {}
S_sw = int(TOTAL * CFG['sweep_frac'])
for lr in CFG['sweep_lrs']:
    key = f'{lr:g}'
    if key in SWEEP:
        print(f'muon lr={key} 이미 있음 → 건너뜀'); continue
    print(f'\n=== 스윕 muon lr={key} ({S_sw} step) ===')
    seed_all(CFG['seed'])
    model = make_model(); opts = make_muon_opts(model, lr, CFG['adam_lr_aux'], CFG['pre_wd'])
    scaler = torch.amp.GradScaler('cuda')
    lr_fn = wsd_fn(S_sw, max(1, int(S_sw * CFG['warmup_frac'])), int(S_sw * (1 - CFG['decay_frac'])))
    log = []
    train_steps(model, opts, scaler, tr_stream, 0, S_sw, lr_fn, CFG['seed'], log, max(1, S_sw // 5), f'sweep {key}', S_sw)
    vl = eval_loss(model, EV_BABY)
    SWEEP[key] = dict(lr=lr, val_loss=vl if math.isfinite(vl) else float('inf'), log=log)
    json.dump(SWEEP, open(sweep_path, 'w'), indent=1)
    del model, opts; torch.cuda.empty_cache()

print('\n===== 스윕 결과 =====')
for k, r in sorted(SWEEP.items(), key=lambda kv: kv[1]['lr']):
    print(f'  muon lr={k}: BabyLM val loss {r["val_loss"]:.4f}')
print('  (어제 L2: 0.01 → 3.3630, 0.02 → 3.4498, 0.04 → 발산)')
BEST = min(SWEEP.values(), key=lambda r: r['val_loss'])
print(f'→ 이번 범위에서 최적: {BEST["lr"]:g}  (loss {BEST["val_loss"]:.4f})')
print('\n참고: 같은 길이로 돌린 AdamW(L1) 스윕 최적값은 1e-3. 그 loss는 L1 노트북의 lr_sweep.json에 있음')


=== 스윕 muon lr=0.0025 (4785 step) ===
[sweep 0.0025] step 957/4785  train 4.369  BabyLM val 3.796 (ppl 44.5)  0.287s/step
[sweep 0.0025] step 1914/4785  train 3.685  BabyLM val 3.611 (ppl 37.0)  0.295s/step
[sweep 0.0025] step 2871/4785  train 3.497  BabyLM val 3.541 (ppl 34.5)  0.295s/step
[sweep 0.0025] step 3828/4785  train 3.469  BabyLM val 3.485 (ppl 32.6)  0.295s/step
[sweep 0.0025] step 4785/4785  train 3.244  BabyLM val 3.354 (ppl 28.6)  0.295s/step

=== 스윕 muon lr=0.005 (4785 step) ===
[sweep 0.005] step 957/4785  train 4.318  BabyLM val 3.780 (ppl 43.8)  0.295s/step
[sweep 0.005] step 1914/4785  train 3.676  BabyLM val 3.609 (ppl 36.9)  0.294s/step
[sweep 0.005] step 2871/4785  train 3.499  BabyLM val 3.540 (ppl 34.5)  0.295s/step
[sweep 0.005] step 3828/4785  train 3.472  BabyLM val 3.486 (ppl 32.7)  0.295s/step
[sweep 0.005] step 4785/4785  train 3.253  BabyLM val 3.348 (ppl 28.4)  0.295s/step

=== 스윕 muon lr=0.01 (4785 step) ===
[sweep 0.01] step 957/4785  train 4.305  Ba